# 🎬 VibeMV Phase 2: True Animation with AnimateDiff

## 🎥 What's Different

**Phase 1:** Static images with zoom/pan effects
**Phase 2:** Actual animated video clips with real motion!

## ✨ Features

- 🎭 Real character movement (walking, dancing, etc.)
- 🌊 Natural motion (rain, waves, wind)
- 📹 True video generation, not animated stills
- 🎨 High quality 512x512 at 24 FPS
- ⚡ Fast enough for Colab free tier

## ⏱️ Time Estimate

- Per scene: ~20-30 seconds
- 47 scenes: ~25-35 minutes total

## 📋 Setup

1. Enable GPU: Runtime → Change runtime type → T4 GPU
2. Run all cells in order
3. Upload timeline JSON
4. Wait for generation
5. Download animated MV!

In [ ]:
# @title ✅ Check GPU
import torch

if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    if torch.cuda.get_device_properties(0).total_memory < 14e9:
        print('   ⚠️  T4 GPU detected - perfect for AnimateDiff!')
else:
    print('❌ No GPU! Enable: Runtime → Change runtime type → T4 GPU')
    raise SystemExit

In [ ]:
# @title 📦 Install AnimateDiff & Dependencies (5-6 minutes)
%%capture

# Core packages
!pip install -q torch torchvision torchaudio
!pip install -q diffusers==0.25.0 transformers accelerate
!pip install -q imageio imageio-ffmpeg
!pip install -q opencv-python pillow
!pip install -q xformers

# AnimateDiff motion module
!pip install -q einops omegaconf

print('✅ AnimateDiff dependencies installed!')

In [ ]:
# @title 📤 Upload Timeline JSON
from google.colab import files
import json

print('📁 Upload your VibeFrame2 timeline JSON...')
uploaded = files.upload()

timeline_file = list(uploaded.keys())[0]
with open(timeline_file, 'r') as f:
    timeline = json.load(f)

is_vibeframe = 'video_prompt' in timeline['scenes'][0]

print(f"\n✅ Loaded {len(timeline['scenes'])} scenes")
print(f"   Duration: {timeline.get('audio_duration', 'N/A')} seconds\n")

for i, scene in enumerate(timeline['scenes'][:5]):
    if is_vibeframe:
        desc = scene.get('description', '')[:60]
        print(f"  {i+1}. {scene['start_time']:.1f}s: {desc}...")

print(f"\n✅ Ready for animation!")

In [ ]:
# @title 🎥 Generate Animated Video Clips with AnimateDiff
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler
from diffusers.utils import export_to_video
import torch
import os

os.makedirs('animated_clips', exist_ok=True)

print('Loading AnimateDiff...')

# Load motion adapter
adapter = MotionAdapter.from_pretrained(
    'guoyww/animatediff-motion-adapter-v1-5-2',
    torch_dtype=torch.float16
)

# Load AnimateDiff pipeline
pipe = AnimateDiffPipeline.from_pretrained(
    'emilianJR/epiCRealism',
    motion_adapter=adapter,
    torch_dtype=torch.float16
).to('cuda')

# Use efficient scheduler
pipe.scheduler = DDIMScheduler.from_config(
    pipe.scheduler.config,
    beta_schedule='linear',
    steps_offset=1
)

# Memory optimizations
pipe.enable_vae_slicing()
pipe.enable_model_cpu_offload()

NEGATIVE = 'bad quality, worse quality, low resolution'

scene_videos = []
print(f"\n🎥 Generating {len(timeline['scenes'])} animated clips...\n")

for i, scene in enumerate(timeline['scenes']):
    # Get prompt
    if 'video_prompt' in scene:
        prompt = scene['video_prompt']
        duration = scene['duration']
    else:
        prompt = scene.get('description', 'cinematic scene')
        duration = scene.get('duration', 4.0)
    
    # Calculate frames (16 frames = ~0.66 seconds at 24fps)
    # We'll generate shorter clips and loop/extend them
    num_frames = 16
    
    print(f"Scene {i+1}/{len(timeline['scenes'])}: {prompt[:70]}...")
    print(f"  Generating {num_frames} frames...")
    
    # Generate animated video
    output = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE,
        num_frames=num_frames,
        num_inference_steps=25,
        guidance_scale=7.5,
        height=512,
        width=512
    )
    
    # Save video clip
    video_path = f"animated_clips/scene_{i:03d}.mp4"
    export_to_video(output.frames[0], video_path, fps=24)
    
    scene_videos.append({
        'path': video_path,
        'duration': duration,
        'frames': num_frames
    })
    
    print(f"  ✅ Saved animated clip\n")
    
    # Memory cleanup every 3 scenes
    if (i + 1) % 3 == 0:
        torch.cuda.empty_cache()

del pipe
del adapter
torch.cuda.empty_cache()

print(f"\n✅ Generated {len(scene_videos)} animated clips!")
print("   Each clip has real motion and animation!")

In [ ]:
# @title 🎬 Stitch All Clips into Final Video
from moviepy.editor import VideoFileClip, concatenate_videoclips
import os

print('🎬 Combining all animated clips...\n')

clips = []
for i, scene_video in enumerate(scene_videos):
    print(f"Loading clip {i+1}/{len(scene_videos)}...")
    
    clip = VideoFileClip(scene_video['path'])
    target_duration = scene_video['duration']
    
    # Loop clip if needed to match scene duration
    if clip.duration < target_duration:
        loops = int(target_duration / clip.duration) + 1
        clip = clip.loop(n=loops).set_duration(target_duration)
    else:
        clip = clip.set_duration(target_duration)
    
    clips.append(clip)

print('\n📹 Concatenating clips...')
final_video = concatenate_videoclips(clips, method='compose')

print('💾 Exporting final video...')
final_video.write_videofile(
    'vibemv_animated.mp4',
    fps=24,
    codec='libx264',
    audio=False
)

# Cleanup
for clip in clips:
    clip.close()
final_video.close()

print('\n✅ Animated MV complete!')
files.download('vibemv_animated.mp4')

---

## ✅ Phase 2 Complete!

**What you got:**
- True animated video (not static images!)
- Real character movement
- Natural motion
- Professional quality

**Optional: Add Audio**
Run the cell below to add your music!

In [ ]:
# @title 🔊 Add Audio (Optional)
from moviepy.editor import VideoFileClip, AudioFileClip
from google.colab import files

print('📁 Upload your audio file...')
uploaded_audio = files.upload()

if uploaded_audio:
    audio_file = list(uploaded_audio.keys())[0]
    
    video = VideoFileClip('vibemv_animated.mp4')
    audio = AudioFileClip(audio_file)
    
    # Trim or loop audio to match video
    if audio.duration > video.duration:
        audio = audio.subclip(0, video.duration)
    
    final = video.set_audio(audio)
    final.write_videofile(
        'vibemv_final_with_audio.mp4',
        fps=24,
        codec='libx264',
        audio_codec='aac'
    )
    
    video.close()
    audio.close()
    final.close()
    
    print('\n✅ Final MV with audio ready!')
    files.download('vibemv_final_with_audio.mp4')